In [1]:
# ==============================================================================
# 0. CHỐT CHẶN MÔI TRƯỜNG: Giải quyết xung đột NumPy 2.x và Bottleneck
# ==============================================================================
import sys
sys.modules['bottleneck'] = None  

import os
import shutil
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
import mlflow.spark
from pyspark.ml.evaluation import RegressionEvaluator

# ==============================================================================
# 1. KHỞI TẠO SPARK SESSION KẾT NỐI LAKEHOUSE (BỔ SUNG LỆNH RESET SESSION)
# ==============================================================================
# CHỐT CHẶN QUAN TRỌNG: Tắt hoàn toàn Session cũ đang chạy ngầm trên Notebook
if 'spark' in locals() or 'spark' in globals():
    print("🔄 Đang giải phóng Spark Session cũ để nạp lại bộ Jars mới...")
    spark.stop()

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("Offline_Model_Testing_Final")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

# Khởi tạo một Session hoàn toàn mới độc lập
spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
print("✅ Khởi tạo Spark Session với đầy đủ Jars Lakehouse thành công!")

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
mlflow.set_tracking_uri("http://mlflow:5000")

# ==============================================================================
# 2. ĐỌC DỮ LIỆU THÔ TỪ CÁC BẢNG GOLD DATABASE
# ==============================================================================
print("⏳ Đang tải dữ liệu từ các bảng trong Lakehouse...")
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")

# Đăng ký temporary view để thực thi Spark SQL
df_fact.createOrReplaceTempView("fact_order")
df_customer.createOrReplaceTempView("dim_customer")
df_product.createOrReplaceTempView("dim_product")
df_time.createOrReplaceTempView("dim_time")
df_location.createOrReplaceTempView("dim_location")

query = """
SELECT 
    f.time_id,
    f.customer_id,
    f.product_id,
    f.location_id,
    f.quantity,
    f.unit_price,        
    f.total_revenue,     

    -- Dim_time
    t.order_date,
    t.year,
    t.month,
    t.day,
    t.quarter,
    t.weekday_name,

    -- Dim_customer
    c.gender,
    c.age_group,
    c.education,
    c.income,
    c.race,
    c.state,             

    -- Dim_product
    p.product_title,
    p.product_category,

    -- Dim_location
    l.state_name purchase_state,
    l.region purchase_region

FROM fact_order f
LEFT JOIN dim_time t     ON f.time_id     = t.time_id
LEFT JOIN dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN dim_product p   ON f.product_id  = p.product_id
LEFT JOIN dim_location l  ON f.location_id = l.location_id
"""
df_fact_full = spark.sql(query)



/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


✅ Khởi tạo Spark Session với đầy đủ Jars Lakehouse thành công!
⏳ Đang tải dữ liệu từ các bảng trong Lakehouse...


In [2]:
# ==============================================================================
# BƯỚC 1: CHẠY LẠI FEATURE ENGINEERING (Đồng bộ cấu trúc dữ liệu Train)
# ==============================================================================
print("⏳ Bước 1: Đang thực thi Feature Engineering gom nhóm theo khách hàng...")
max_date_val = df_fact_full.select(F.max("order_date")).collect()[0][0]

df_enhanced = (
    df_fact_full.groupBy("customer_id")
    .agg(
        F.sum("total_revenue").alias("total_spend"),              
        F.countDistinct("time_id").alias("total_orders"),         
        F.countDistinct("product_category").alias("category_diversity"), 
        F.avg("quantity").alias("avg_items_per_line"),            
        F.min("year").alias("first_year"),                        
        F.max("year").alias("last_year"),                         
        F.datediff(F.lit(max_date_val), F.max("order_date")).alias("recency"),
        F.first("age_group").alias("age_group")
    )
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    
    # Target & Biến bổ trợ
    .withColumn("avg_orders_per_year", F.col("total_orders") / F.col("years_active"))
    .withColumn("avg_order_value", F.col("total_spend") / F.col("total_orders"))
    
    # Biến đổi Logarit
    .withColumn("total_spend_log", F.log1p("total_spend"))
    .withColumn("avg_order_value_log", F.log1p("avg_order_value"))
    .withColumn("recency_log", F.log1p("recency"))
    .withColumn("avg_items_log", F.log1p("avg_items_per_line"))
    
    # Điền khuyết thiếu
    .na.fill("Unknown", ["age_group"])
    .na.fill(0)
)



⏳ Bước 1: Đang thực thi Feature Engineering gom nhóm theo khách hàng...


In [3]:
# ==============================================================================
# BƯỚC 2: TÁI TẠO TẬP DỮ LIỆU TEST_DF (Sử dụng seed=42 đồng nhất)
# ==============================================================================
print("⏳ Bước 2: Phân tách dữ liệu Train/Test mẫu độc lập...")
train_df, test_df = df_enhanced.randomSplit([0.8, 0.2], seed=42)

# ==============================================================================
# BƯỚC 3: TRUY XUẤT MÔ HÌNH PRODUCTION TỪ CENTRAL MODEL REGISTRY
# ==============================================================================
print("⚡ Bước 3: Đang kết nối MLflow Server tải mô hình Production...")
model_uri = "models:/XGBoost_Tweedie_Final/Production"
model = mlflow.spark.load_model(model_uri)

# ==============================================================================
# BƯỚC 4: THỰC THI DỰ ĐOÁN (PREDICT) TRÊN TẬP TEST
# ==============================================================================
print("🚀 Bước 4: Đang chạy dự đoán mô hình trên tập Test...")
test_pred = model.transform(test_df)



⏳ Bước 2: Phân tách dữ liệu Train/Test mẫu độc lập...
⚡ Bước 3: Đang kết nối MLflow Server tải mô hình Production...


2026/07/07 13:29:18 INFO mlflow.spark: URI 'models:/XGBoost_Tweedie_Final/Production/sparkml' does not point to the current DFS.
2026/07/07 13:29:18 INFO mlflow.spark: File 'models:/XGBoost_Tweedie_Final/Production/sparkml' not found on DFS. Will attempt to upload the file.


🚀 Bước 4: Đang chạy dự đoán mô hình trên tập Test...


In [4]:
# ==============================================================================
# BƯỚC 5: HIỂN THỊ ĐỐI CHIẾU KẾT QUẢ DỰ ĐOÁN THỰC TẾ (STYLING BẢNG BIỂU)
# ==============================================================================
print("\n📊 Bước 5: Đang khởi tạo bảng đối chiếu hiệu năng HTML...")

# 1. Lấy 20 dòng đầu tiên của tập kết quả và chuyển sang Pandas DataFrame
preview_df = (
    test_pred.select(
        "customer_id",
        F.round("avg_orders_per_year", 2).alias("Giá Trị Thực Tế"),
        F.round("prediction", 2).alias("Mô Hình AI Dự Báo")
    )
    .limit(20)
    .toPandas()
)

# 2. Tính toán thêm cột Độ Lệch Tuyệt Đối để dễ dàng đánh giá sai số từng dòng
preview_df["Sai Số Tuyệt Đối (Lệch)"] = (preview_df["Mô Hình AI Dự Báo"] - preview_df["Giá Trị Thực Tế"]).abs()

# 3. Sử dụng Pandas Styler để định dạng giao diện chuyên nghiệp
styled_preview = (
    preview_df.style
    # Định dạng hiển thị số lẻ 2 chữ số thập phân
    .format({
        "Giá Trị Thực Tế": "{:,.2f}",
        "Mô Hình AI Dự Báo": "{:,.2f}",
        "Sai Số Tuyệt Đối (Lệch)": "{:,.2f}"
    })
    # Tô màu nền dải gradient cho cột Sai Số để nhận diện nhanh những dòng AI dự đoán lệch nhiều nhất
    .background_gradient(subset=["Sai Số Tuyệt Đối (Lệch)"], cmap="Reds")
    # Cấu hình thuộc tính CSS hiển thị căn lề và cỡ chữ
    .set_properties(**{
        'text-align': 'right',
        'font-size': '11pt',
        'font-family': 'Calibri, sans-serif'
    })
    # Cấu hình tiêu đề cột (Header) có màu nền xám nhạt làm nổi bật bảng dữ liệu
    .set_table_styles([
        {
            'selector': 'th', 
            'props': [
                ('text-align', 'center'), 
                ('background-color', '#f0f2f6'),
                ('font-weight', 'bold')
            ]
        },
        {
            'selector': 'td:first-child', 
            'props': [('text-align', 'center')] # Cho cột customer_id ra giữa bảng
        }
    ])
    # Đặt tên tiêu đề lớn cho bảng
    .set_caption("Bảng Đối Chiếu Chi Tiết Giá Trị Thực Tế và Kết Quả Dự Báo Từ Mô Hình Production (XGBoost Tweedie)")
)

# Hiển thị bảng biểu định dạng HTML trực tiếp trên Notebook
styled_preview


📊 Bước 5: Đang khởi tạo bảng đối chiếu hiệu năng HTML...


,customer_id,Giá Trị Thực Tế,Mô Hình AI Dự Báo,Sai Số Tuyệt Đối (Lệch)
0,R_0Arj0ePpTnReV1v,14.33,14.93,0.60
1,R_0Hzrb8hEmXavTk5,41.33,41.50,0.17
2,R_0TXaBMuEhEzmPVD,3.67,4.29,0.62
3,R_0e9mwIgnQ7UBcGJ,13.80,15.08,1.28
4,R_10CAlrx7dbJty9R,168.17,163.96,4.21
5,R_10PDLObOugOU8tc,8.80,9.76,0.96
6,R_10YhHZtIc1xIXOM,5.00,4.88,0.12
7,R_117LDoOnJxdPvkb,28.40,28.36,0.04
8,R_12bxXZmV1h6iVAL,14.50,14.87,0.37
9,R_12kYTBVH8kS49hc,17.40,17.38,0.02


In [5]:
# ==============================================================================
# BƯỚC 6: TỰ ĐỘNG TRUY VẤN METRICS GỐC VÀ KIỂM TRA ĐỘ TRÙNG KHỚP NGHIỆM THU
# ==============================================================================
print("\n📈 Bước 6: Central Registry đang tự động kết nối Tracking Server để lấy Baseline...")

from mlflow.tracking import MlflowClient

# 1. Khởi tạo Mlflow Client để tra cứu thông tin bắc cầu
client = MlflowClient()
model_name = "XGBoost_Tweedie_Final"

try:
    # Tự động lấy thông tin phiên bản mô hình đang nằm ở Stage Production
    production_version = client.get_latest_versions(model_name, stages=["Production"])[0]
    run_id = production_version.run_id
    
    # Truy vấn ngược lại kho dữ liệu Tracking để lấy bộ chỉ số logged lúc train
    run_data = client.get_run(run_id).data
    
    # Gán tự động vào bộ thông số nền (Lấy đúng các key mae_test, r2_test, rmse_test bạn đã log)
    baseline_metrics = {
        "r2": round(run_data.metrics.get("r2_test"), 4),
        "mae": round(run_data.metrics.get("mae_test"), 4),
        "rmse": round(run_data.metrics.get("rmse_test"), 4)
    }
    print(f"🎯 Kết nối thành công! Đã tìm thấy Run ID gốc: {run_id}")
    
except Exception as e:
    print(f"⚠️ Không thể lấy metrics tự động từ MLflow (Lỗi: {e}). Hệ thống chuyển sang dùng Baseline dự phòng.")
    # Cấu hình dự phòng trường hợp kết nối mạng nội bộ giữa các service bị nghẽn
    baseline_metrics = {"r2": 0.9807, "mae": 1.3586, "rmse": 4.2082}

# 2. Tiến hành tính toán hiệu năng hiện tại và so sánh tự động
metrics_summary = []

for metric in ['r2', 'mae', 'rmse']:
    evaluator = RegressionEvaluator(
        labelCol="avg_orders_per_year",
        predictionCol="prediction",
        metricName=metric
    )
    
    # Tính toán điểm số thực tế trên tập Test hiện tại
    current_score = round(evaluator.evaluate(test_pred), 4)
    original_score = baseline_metrics[metric]
    
    # Hệ thống tự động thẩm định tính toàn vẹn
    status = "✅ Khớp" if current_score == original_score else "❌ Lệch"
    
    metrics_summary.append({
        "Chỉ số": metric.upper(),
        "Lúc huấn luyện (Tự động từ MLflow)": original_score,
        "Sau load từ Registry (Môi trường Test)": current_score,
        "Kết quả nghiệm thu": status
    })

# 3. Đóng gói kết quả sang Pandas DataFrame để làm đẹp giao diện HTML
df_metrics_report = pd.DataFrame(metrics_summary)

styled_metrics = (
    df_metrics_report.style
    .set_properties(**{
        'text-align': 'center',
        'font-size': '11pt',
        'font-family': 'Calibri, sans-serif'
    })
    .set_table_styles([
        {
            'selector': 'th', 
            'props': [
                ('text-align', 'center'), 
                ('background-color', '#f0f2f6'),
                ('font-weight', 'bold'),
                ('padding', '8px')
            ]
        },
        {
            'selector': 'td',
            'props': [('padding', '8px')]
        }
    ])
    # Tự động highlight màu xanh làm nổi bật các chỉ số đạt chuẩn nghiệm thu
    .map(lambda v: 'background-color: #e6f4ea; color: #137333; font-weight: bold;' if v == "✅ Khớp" else '', subset=["Kết quả nghiệm thu"])
    .set_caption("Bảng Đối Chiếu Chỉ Số Hiệu Năng Tự Động Từ Hệ Thống Central Registry (Automated MLOps Verification Report)")
)

# Hiển thị bảng nghiệm thu tự động hoàn toàn
styled_metrics


📈 Bước 6: Central Registry đang tự động kết nối Tracking Server để lấy Baseline...


/tmp/ipykernel_796/21589793.py:14: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  production_version = client.get_latest_versions(model_name, stages=["Production"])[0]


🎯 Kết nối thành công! Đã tìm thấy Run ID gốc: 02b6dcabda934aeabf308d3f9a132f8f


,Chỉ số,Lúc huấn luyện (Tự động từ MLflow),Sau load từ Registry (Môi trường Test),Kết quả nghiệm thu
0,R2,0.981900,0.981900,✅ Khớp
1,MAE,1.289700,1.289700,✅ Khớp
2,RMSE,4.062600,4.062600,✅ Khớp


In [ ]:
xong